# ProofAgent™ — Judge-led evaluation (Google Colab)

This notebook runs a **judge-led** evaluation: the AI Judge asks one **question per turn**; you submit an **agent answer** until all turns complete, then **finalize** and read the **report**.

## Definitions
| Term | Meaning |
|------|--------|
| **Run** | One job; you get a `run_id` from `start_run`. |
| **Turn** | One judge question + one agent answer (index starts at **1**). |
| **Poll until ready** | Wait until the run can accept questions (`poll_until_ready`). |
| **Finalize** | Call after all turns so the backend can score. |
| **Transcript** | Per-turn rows in the report: question, answer, optional conductor notes. |

## Setup order
1. **Install** the SDK (pip from Git URL or upload this repo).
2. **Secrets**: add `PROOFAGENT_API_KEY` (and optional `OPENAI_API_KEY`) in Colab secrets, or paste in the auth cell for quick tests only.
3. **Run** all cells.

You will see **live progress** (turn *i* / *N*, bar) during the loop, then **aggregate scores** and **turn-level details** from the report.


## 1) Install dependencies and SDK

**Option A — public Git URL** (replace with your repository URL if you forked the SDK):
```
pip install "git+https://github.com/YOUR_ORG/Proofagent-python-sdk.git"
```

**Option B — upload** the `Proofagent-python-sdk` folder to Colab (`Files` sidebar), then set `SDK_ROOT` in the next cell to `/content/Proofagent-python-sdk` (or your path) and `pip install -e` that path.


In [ ]:
# --- Install (edit GIT_URL or use upload + pip install -e) ---
import subprocess, sys, os

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "httpx"])

# Default: install from git. Set GIT_URL = None to skip and use SDK_ROOT only.
GIT_URL = os.environ.get("PROOFAGENT_SDK_GIT", "")  # e.g. "git+https://github.com/org/Proofagent-python-sdk.git"
SDK_ROOT = os.environ.get("PROOFAGENT_SDK_ROOT", "/content/Proofagent-python-sdk")

if GIT_URL:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", GIT_URL])
elif os.path.isfile(os.path.join(SDK_ROOT, "pyproject.toml")):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", SDK_ROOT])
else:
    print("Set GIT_URL to your SDK git URL, or upload the repo and set PROOFAGENT_SDK_ROOT / SDK_ROOT.")

print("pip install step done.")


## 2) API keys

Recommended: **Colab** → **Secrets** (key icon) → add `PROOFAGENT_API_KEY` and optionally `OPENAI_API_KEY`.

Or set environment variables in **Runtime → Change runtime type** is not needed; use the code below.


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["PROOFAGENT_API_KEY"] = userdata.get("PROOFAGENT_API_KEY")
    try:
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        pass
except ImportError:
    pass

# Fallback for local testing only — do not commit real keys
if not os.environ.get("PROOFAGENT_API_KEY", "").strip():
    os.environ["PROOFAGENT_API_KEY"] = ""  # paste apk_live_… here if not using userdata

assert os.environ.get("PROOFAGENT_API_KEY", "").strip(), "Set PROOFAGENT_API_KEY (secret or above)"
print("API key loaded:", bool(os.environ.get("PROOFAGENT_API_KEY")))


In [ ]:
import asyncio
import os
import sys
from pathlib import Path

from proofagent import ProofAgentClient

# report_helpers: same repo as SDK — add examples/ to path
_roots = [Path.cwd(), Path("/content/Proofagent-python-sdk"), Path(".")]
for r in _roots:
    ex = r / "examples" / "report_helpers.py"
    if ex.is_file():
        sys.path.insert(0, str(r / "examples"))
        break

from report_helpers import (
    print_live_turn_banner,
    print_full_evaluation_report,
)


async def main():
    async with ProofAgentClient.from_env() as client:
        ctx = await client.get_project_context()
        bill = await client.get_billing()
        proj = ctx["data"]["project"]
        cap = int(bill["data"]["max_turns_per_run"])
        turns = min(int(proj.get("turn_count", 5)), cap)
        print("Project:", proj["name"], "| domain:", proj["domain"])
        print("Effective turns:", turns, "(capped by billing)")

        byo = os.environ.get("OPENAI_API_KEY", "").strip()
        run = await client.start_run(
            turn_count=turns,
            llm_api_key=byo or None,
            llm_provider="openai" if byo else None,
            llm_model="gpt-4o-mini" if byo else None,
            agent_role="Helpful support assistant",
            tools=[{"name": "policy_lookup", "description": "Policy clauses"}],
            internal_agents=[{"id": "p1", "role": "policy", "description": "policy helper"}],
        )
        rid = run["data"]["run_id"]
        print("Run id:", rid)

        st = await client.poll_until_ready(rid)
        total = int(st["data"]["total_turns"])
        print(f"Interactive phase: {total} turn(s)")

        for i in range(1, total + 1):
            print_live_turn_banner(i, total, phase="fetching_question")
            q = await client.get_next_question(rid)
            jq = q["data"]["judge_question"]
            ans = f"Answer {i}: We follow policy. Q: {jq}"
            print_live_turn_banner(i, total, phase="submitting")
            await client.submit_turn(rid, turn_index=i, agent_answer=ans)
            print(f"  ✓ Submitted {i}/{total}")

        print("\nFinalizing…")
        await client.finalize(rid)
        report = await client.get_report(rid)
        print_full_evaluation_report(report)


await main()
